# 1.1. Install and import dependencies

In [1]:
import os
from pathlib import Path

# Ensure working directory is always the repo root regardless of where
# the notebook is launched from (e.g. notebooks/ subdirectory)
repo_root = Path(__file__).parent.parent if "__file__" in dir() else Path.cwd().parent
if (repo_root / "pyproject.toml").exists():
    os.chdir(repo_root)

print(f"Working directory: {Path.cwd()}")

Working directory: /home/dima/ml_engineering


In [2]:
# Dependencies are managed via the local .venv virtual environment.
# To set up: python3 -m venv .venv && .venv/bin/pip install -r requirements.txt
# All required packages are already installed in .venv.

In [3]:
pyproject_content = """[tool.poetry]
name = "lab_1_example"
description = "Training pipeline for deep learning model"
authors = ["Name Surname <name.surname@gmail.com>"]
version = "0.01"

[tool.poetry.dependencies]
python = "~3.12"
torch = ">=2.0"
torchvision = ">=0.15"
tqdm = ">=4.64"
matplotlib = ">=3.6"
numpy = ">=1.22"
pyyaml = ">=6.0"
scipy = ">=1.10"
pandas = ">=2.0"
scikit-learn = ">=1.0"
Pillow = ">=9.0"

[tool.poetry.dev-dependencies]
mypy = ">=0.991"
ruff = ">=0.0.254"
black = ">=23.1.0"
isort = ">=5.12.0"

[build-system]
requires = ["poetry-core>=1.0.0"]
build-backend = "poetry.core.masonry.api" """

# Write the content to pyproject.toml
with open("pyproject.toml", "w") as file:
    file.write(pyproject_content)

print("pyproject.toml written.")
print("To install dependencies, run: python3 -m venv .venv && .venv/bin/pip install -r requirements.txt")

pyproject.toml written.
To install dependencies, run: python3 -m venv .venv && .venv/bin/pip install -r requirements.txt


In [4]:
import pickle
import zipfile
import tarfile
import os
import logging
import requests

from pathlib import Path
from typing import Optional, Tuple, Dict, Any, Union, List

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Dataset
from torchvision.io import read_image
from torchvision import transforms

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# hack to fix logging in colab https://stackoverflow.com/a/74121821/11709937
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(level=logging.INFO)

# 1.2 Data download

In [6]:
def download_and_extract(url: str, save_dir: str, filename: Optional[str] = None) -> str:
    """
    Downloads a file from the given URL and extracts it if it is an archive.

    Args:
    - url (str): The URL of the file to download.
    - save_dir (str): Directory where the file will be saved and extracted.
    - filename (Optional[str]): Optional filename to save the file with. If None, uses the filename from the URL.

    Returns:
    - str: The full path to the directory containing the extracted files or the downloaded file.
    """
    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)

    if filename is None:
        filename = url.split("/")[-1]

    file_path = save_path / filename
    result_path = str(file_path)

    # Check if file already exists
    if file_path.exists():
        logging.info(f"File '{filename}' already exists in '{save_dir}'. Skipping download.")
    else:
        logging.info(f"Downloading file '{filename}' from '{url}'...")
        try:
            response = requests.get(url)
            response.raise_for_status()
            with open(file_path, 'wb') as f:
                f.write(response.content)
            logging.info(f"Download successful. File saved to: '{file_path}'")

            # Extract the file if it's an archive
            if filename.endswith(".zip"):
                with zipfile.ZipFile(file_path, 'r') as zip_ref:
                    zip_ref.extractall(save_path)
                file_path.unlink()
                result_path = str(save_path)
            elif (
                  filename.endswith(".tar.gz")
                  or filename.endswith(".tgz")
                  or filename.endswith(".gz")
                ):
                with tarfile.open(file_path, 'r:gz') as tar_ref:
                    tar_ref.extractall(save_path)
                file_path.unlink()
                result_path = str(save_path)
            elif filename.endswith(".tar"):
                with tarfile.open(file_path, 'r') as tar_ref:
                    tar_ref.extractall(save_path)
                file_path.unlink()
                result_path = str(save_path)
            else:
                logging.info(f"File '{filename}' is not an archive. No extraction needed.")

        except requests.RequestException as e:
            logging.error(f"Error downloading file from {url}: {str(e)}")
            raise
        except (zipfile.BadZipFile, tarfile.ReadError) as e:
            logging.error(f"Error extracting file '{filename}': {str(e)}")
            raise

    return result_path

# 1.3 Data ingestion

In [7]:
def train_test_split(
    data: pd.DataFrame,
    test_size: Union[float, int] = 0.25,
    random_state: Union[int, None] = None
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split arrays or matrices into random train and test subsets.

    Parameters:
    data : pd.DataFrame
         The input data to split.
    test_size : float, int, or None, optional (default=0.25)
        If float, should be between 0.0 and 1.0 and represent the proportion of the dataset to include in the test split.
        If int, represents the absolute number of test samples.
    random_state : int or None, optional (default=None)
        If int, random_state is the seed used by the random number generator.

    Returns:
    Tuple containing:
        - data_train: pd.DataFrame
        - data_test: pd.DataFrame
    """
    if random_state is not None:
        np.random.seed(random_state)

    n_samples = len(data)
    if isinstance(test_size, float):
        test_size = int(n_samples * test_size)

    indices = np.random.permutation(n_samples)
    test_indices = indices[:test_size]
    train_indices = indices[test_size:]

    data_train = data.iloc[train_indices]
    data_test = data.iloc[test_indices]

    return data_train, data_test


# CIFAR-10 class names indexed 0-9
CIFAR10_CLASSES: List[str] = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]


def load_labels(labels_path: str) -> pd.DataFrame:
    """
    Load CIFAR-10 dataset from the extracted directory, save images as JPEGs,
    and return a DataFrame with image_path and label columns.

    CIFAR-10 is distributed as Python pickle batch files. This function reads
    all batch files, saves each image to disk as a JPEG, and returns a DataFrame
    suitable for use with ImageDataset.

    Args:
    - labels_path (str): Path to the extracted CIFAR-10 directory
      (i.e. the 'cifar-10-batches-py' folder).

    Returns:
    - pd.DataFrame: DataFrame with columns 'image_path' (str) and 'label' (int, 0-9).
    """
    cifar_dir = Path(labels_path)
    images_out_dir = cifar_dir.parent / "images"
    images_out_dir.mkdir(parents=True, exist_ok=True)

    # Create class subdirectories
    for class_name in CIFAR10_CLASSES:
        (images_out_dir / class_name).mkdir(parents=True, exist_ok=True)

    batch_files = sorted([
        f for f in cifar_dir.iterdir()
        if f.name.startswith("data_batch") or f.name == "test_batch"
    ])

    records: List[Dict[str, Any]] = []
    img_counter = 0

    for batch_file in batch_files:
        logging.info(f"Loading batch: {batch_file.name}")
        with open(batch_file, "rb") as f:
            batch = pickle.load(f, encoding="bytes")

        images_raw: np.ndarray = batch[b"data"]   # shape: (N, 3072)
        labels_raw: List[int] = batch[b"labels"]

        for img_flat, label in zip(images_raw, labels_raw):
            # CIFAR-10 stores images as 3x32x32 in row-major order
            img_array = img_flat.reshape(3, 32, 32).transpose(1, 2, 0)  # HWC
            class_name = CIFAR10_CLASSES[label]
            img_filename = f"image_{img_counter:05d}.jpg"
            img_path = images_out_dir / class_name / img_filename

            if not img_path.exists():
                Image.fromarray(img_array).save(str(img_path))

            records.append({"image_path": str(img_path.absolute()), "label": label})
            img_counter += 1

    logging.info(f"Loaded {img_counter} images from CIFAR-10 batches.")
    return pd.DataFrame(records)


def process_data(
    images_dir: str,
    labels_path: str,
    config: Dict[str, Any]
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Process the data by loading labels, splitting into train/validation/test sets.

    Args:
    - images_dir (str): Not used for CIFAR-10 (images are extracted inside load_labels).
    - labels_path (str): Path to the extracted cifar-10-batches-py directory.
    - config (Dict[str, Any]): Configuration dictionary with test_size, val_size, random_state.

    Returns:
    - Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]: Training, validation, and testing DataFrames.
    """
    # Load labels and save images to disk
    labels_df = load_labels(labels_path)

    # Split into train+val and test
    train_val_df, test_df = train_test_split(
        labels_df,
        test_size=config.get("test_size", 0.2),
        random_state=config.get("random_state", 42)
    )

    # Further split train+val into train and validation
    train_df, val_df = train_test_split(
        train_val_df,
        test_size=config.get("val_size", 0.2),
        random_state=config.get("random_state", 42)
    )

    logging.info(
        f"Prepared 3 data splits — train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}"
    )
    return train_df, val_df, test_df

# 1.4 Training loop

## 1.4.1 Data loaders

In [8]:
class ImageDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform: Optional[Any] = None) -> None:
        self.dataframe = dataframe
        self.transform = transform
        # CIFAR-10 standard normalization values
        self.transform_default = transforms.Compose([
            transforms.Resize([32, 32]),
            transforms.Normalize(
                mean=[0.4914, 0.4822, 0.4465],
                std=[0.2023, 0.1994, 0.2010]
            )
        ])

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        img_path = self.dataframe.iloc[idx]["image_path"]
        image = read_image(img_path).float() / 255.0
        label = int(self.dataframe.iloc[idx]["label"])

        if self.transform:
            image = self.transform(image)
        else:
            image = self.transform_default(image)

        return image, label


# Augmentation transform for training
train_transform = transforms.Compose([
    transforms.Resize([32, 32]),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2023, 0.1994, 0.2010]
    )
])

# Standard transform for validation and test (no augmentation)
eval_transform = transforms.Compose([
    transforms.Resize([32, 32]),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2023, 0.1994, 0.2010]
    )
])


def create_data_loader(
    images_dir: str,
    df: pd.DataFrame,
    config: Dict[str, Any]
) -> DataLoader:
    """
    Create a data loader for a dataset.

    Args:
    - images_dir (str): Directory containing the images (unused for CIFAR-10, paths are in df).
    - df (pd.DataFrame): DataFrame containing the dataset with image_path and label columns.
    - config (Dict[str, Any]): Configuration dictionary containing batch_size, num_workers, transform.

    Returns:
    - DataLoader: DataLoader for the dataset.
    """
    transform: Optional[Any] = config.get("transform", None)
    batch_size: int = config.get("batch_size", 128)
    num_workers: int = config.get("num_workers", 2)

    dataset = ImageDataset(df, transform=transform)
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)

    return data_loader

## 1.4.2. Model, loss and optimizers

In [9]:
class CifarCNN(nn.Module):
    """
    CNN model for CIFAR-10 image classification.

    Architecture:
    - 3 convolutional blocks with BatchNorm and ReLU activations
    - MaxPool2d after blocks 2 and 3 to reduce spatial dimensions
    - Classifier head with Dropout for regularization

    Input: (N, 3, 32, 32) — batch of 32x32 RGB images
    Output: (N, n_classes) — class logits
    """

    def __init__(self, n_classes: int) -> None:
        super(CifarCNN, self).__init__()
        self.features = nn.Sequential(
            # Block 1: 3 -> 32 channels, 32x32 -> 32x32
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            # Block 2: 32 -> 64 channels, 32x32 -> 16x16
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            # Block 3: 64 -> 128 channels, 16x16 -> 8x8
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(128 * 8 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

## 1.4.3. Training loop function

In [10]:
def train_model(
        model: nn.Module,
        train_loader: DataLoader,
        val_loader: DataLoader,
        loss_function: nn.Module,
        optimizer: optim.Optimizer,
        num_epochs: int,
        device: torch.device,
        save_path: Path = Path("best_model.pth")
        ) -> Path:

    model.to(device)
    best_val_loss: float = float('inf')
    best_model_path: Path = Path()

    for epoch in range(num_epochs):
        model.train()  # Set the model to training mode
        for batch_inputs, batch_targets in train_loader:
            batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

            # Forward pass
            outputs: torch.Tensor = model(batch_inputs)
            loss: torch.Tensor = loss_function(outputs, batch_targets)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        logging.info(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")


        # Validation step
        model.eval()  # Set the model to evaluation mode

        with torch.no_grad():

            val_loss: float = 0.0
            for val_inputs, val_targets in val_loader:

                val_inputs, val_targets = val_inputs.to(device), val_targets.to(device)
                val_outputs: torch.Tensor = model(val_inputs)
                val_loss += loss_function(val_outputs, val_targets).item()

            val_loss /= len(val_loader)
            logging.info(f"Epoch {epoch+1}/{num_epochs}, Validation Loss: {val_loss:.4f}")

            # Save the best model
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_path = save_path
                torch.save(model.state_dict(), best_model_path)

                logging.info(f"Best model saved with validation loss: {best_val_loss:.4f}")

    logging.info("Training complete.")

    return best_model_path

# 1.5 Test loop

In [11]:
def test_model(
        model: nn.Module,
        test_loader: DataLoader,
        loss_function: nn.Module,
        device: torch.device
        ) -> Dict[str, float]:
    """
    Test a trained model on a test dataset and compute test metrics.

    Args:
    - model (nn.Module): Trained PyTorch model.
    - test_loader (DataLoader): DataLoader for the test dataset.
    - loss_function (nn.Module): Loss function for computing the loss.
    - device (torch.device): Device (CPU or GPU) on which to run the evaluation.

    Returns:
    - Dict[str, float]: Dictionary containing loss, accuracy, precision, recall, and f1 score.
    """
    model.eval()
    test_loss = 0.0
    all_preds: List[int] = []
    all_targets: List[int] = []

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            outputs = model(inputs)
            loss = loss_function(outputs, targets)
            test_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_targets.extend(targets.cpu().tolist())

    test_loss /= len(test_loader)
    acc = accuracy_score(all_targets, all_preds)
    prec = precision_score(all_targets, all_preds, average="weighted", zero_division=0)
    rec = recall_score(all_targets, all_preds, average="weighted", zero_division=0)
    f1 = f1_score(all_targets, all_preds, average="weighted", zero_division=0)

    metrics = {
        "loss": test_loss,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
    }

    logging.info(
        f"Test Results — Loss: {test_loss:.4f} | Accuracy: {acc:.4f} | "
        f"Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}"
    )

    return metrics

In [12]:
def main() -> None:
    # Configuration — all pipeline parameters in one place
    config: Dict[str, Any] = {
        "test_size": 0.2,
        "val_size": 0.2,
        "random_state": 42,
        "lr": 0.001,
        "batch_size": 128,
        "num_epochs": 20,
        "num_workers": 2,
        "n_classes": 10,
        "save_path": Path("best_model.pth"),
    }

    # Step 1: Download CIFAR-10 dataset
    cifar_url = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"
    data_dir = download_and_extract(cifar_url, "./data")
    cifar_batches_dir = str(Path(data_dir) / "cifar-10-batches-py")

    # Step 2: Process data — unpack images and split into train/val/test
    train_df, val_df, test_df = process_data(
        images_dir=str(Path(data_dir) / "images"),
        labels_path=cifar_batches_dir,
        config=config
    )

    # Step 3: Create data loaders (training uses augmentation, val/test do not)
    train_loader = create_data_loader(
        "",
        train_df,
        {**config, "transform": train_transform}
    )
    val_loader = create_data_loader(
        "",
        val_df,
        {**config, "transform": eval_transform}
    )
    test_loader = create_data_loader(
        "",
        test_df,
        {**config, "transform": eval_transform}
    )

    # Step 4: Define model, loss function, optimizer
    model = CifarCNN(n_classes=config["n_classes"]).to(device)
    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config["lr"])

    # Step 5: Train model
    best_model_path = train_model(
        model,
        train_loader,
        val_loader,
        loss_function,
        optimizer,
        num_epochs=config["num_epochs"],
        device=device,
        save_path=config["save_path"]
    )

    # Step 6: Load best model and evaluate on test set
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    metrics = test_model(model, test_loader, loss_function, device)

    logging.info(
        f"Final metrics — Accuracy: {metrics['accuracy']:.4f}, "
        f"Precision: {metrics['precision']:.4f}, "
        f"Recall: {metrics['recall']:.4f}, "
        f"F1: {metrics['f1']:.4f}"
    )

In [13]:
main()

INFO:root:Downloading file 'cifar-10-python.tar.gz' from 'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz'...
INFO:root:Download successful. File saved to: 'data/cifar-10-python.tar.gz'
INFO:root:Loading batch: data_batch_1
/tmp/ipykernel_26273/3599044242.py:82: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  batch = pickle.load(f, encoding="bytes")
INFO:root:Loading batch: data_batch_2
INFO:root:Loading batch: data_batch_3
INFO:root:Loading batch: data_batch_4
INFO:root:Loading batch: data_batch_5
INFO:root:Loading batch: test_batch
INFO:root:Loaded 60000 images from CIFAR-10 batches.
INFO:root:Prepared 3 data splits — train: 38400, val: 9600, test: 12000
INFO:root:Epoch 1/20, Loss: 1.6155
INFO:root:Epoch 1/20, Validation Loss: 1.4345
INFO:root:Best model saved with validation loss: 1.4345
INFO:root:Epoch 2/20, Loss: 1.1901
INFO:root:Epoch 2